*0.3 Classical NLP*

# Break → Fix

**The situation.** A team ships the TF-IDF router from item 15. Offline accuracy: 88%. In production: close to a coin flip, from day one. The training script saved the classifier. The serving code — written later, in another repo — builds a `TfidfVectorizer`, calls `fit_transform` on each incoming batch, and passes the result to the saved classifier.

**The bug.** `fit_transform` learns a *new* vocabulary from the batch. Column 0 now means whatever word happened to sort first in that batch, not the word it meant in training. The classifier's weights are applied to the wrong words. The shapes are made to match by `max_features`, so nothing errors.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**The broken version.**

In [2]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

topics = ["sci.med", "sci.space", "rec.autos"]
train = fetch_20newsgroups(
    subset="train", categories=topics, remove=("headers", "footers", "quotes"), random_state=0
)
test = fetch_20newsgroups(
    subset="test", categories=topics, remove=("headers", "footers", "quotes"), random_state=0
)

# Training script: fit the vectorizer, train the classifier, save ONLY the classifier.
train_vectorizer = TfidfVectorizer(max_features=5_000, min_df=2)
classifier = LogisticRegression(C=10, max_iter=3000).fit(
    train_vectorizer.fit_transform(train.data), train.target
)
print(
    
        f"offline evaluation: "
        f"{classifier.score(train_vectorizer.transform(test.data), test.target):.1%}"
    
)

# BREAK — serving code: a fresh vectorizer, fitted on the incoming batch.
serving_vectorizer = TfidfVectorizer(max_features=5_000, min_df=2)
batch_features = serving_vectorizer.fit_transform(test.data)  # new vocabulary, new column order
broken_accuracy = classifier.score(batch_features, test.target)
print(f"production:         {broken_accuracy:.1%}   ← BREAK (no error, wrong columns)")
print(
    "column 100 means",
    repr(train_vectorizer.get_feature_names_out()[100]),
    "in training and",
    repr(serving_vectorizer.get_feature_names_out()[100]),
    "in serving",
)

offline evaluation: 85.6%
production:         33.7%   ← BREAK (no error, wrong columns)
column 100 means '222' in training and '24' in serving


**The fix.** Save the vectorizer *with* the classifier as one pipeline object, and only ever call `predict` on it. Serving code cannot re-fit what it does not construct.

In [3]:
import pickle

from sklearn.pipeline import make_pipeline

pipeline = make_pipeline(
    TfidfVectorizer(max_features=5_000, min_df=2), LogisticRegression(C=10, max_iter=3000)
)
pipeline.fit(train.data, train.target)
artefact = pickle.dumps(pipeline)  # this one object is what ships

served = pickle.loads(artefact)  # serving code: load, predict, nothing else
fixed_accuracy = served.score(test.data, test.target)
print(f"production:         {fixed_accuracy:.1%}   ← FIX")
assert fixed_accuracy > broken_accuracy + 0.2

production:         85.6%   ← FIX


**Reading the output.** Same classifier, same data. With a re-fitted vectorizer, accuracy falls to around chance; with the fitted vectorizer inside the shipped pipeline, production matches the offline number.

**How you notice it.** Offline and production accuracy disagree from day one; serving code contains `fit` or `fit_transform`; the column for a word differs between the two vectorizers.

**Watch out**
- `fit` belongs in training only. In serving, the only verbs are `load` and `predict`.
- Ship the whole pipeline (every transform + the model) as one artefact — same rule as stemming, lowercasing and stopwords.
- The same bug exists for embeddings: an index built with one model and queried with another (0.2 Break → Fix).